# Phase 2 — PySpark DataFrame API

**Duration:** Weeks 3–4 | **Goal:** Fluency in the day-to-day language of data engineering.

---

### What We'll Cover

| Area | Key Skills |
| --- | --- |
| Creating DataFrames | From files (CSV/JSON/Parquet), tables, and schemas |
| Schemas | `StructType`; explicit schema vs. `inferSchema` — why explicit wins in production |
| Select & Filter | `select`, `filter`/`where`, `col`, column expressions |
| Transformations | `withColumn`, `withColumnRenamed`, `drop`, `cast` |
| Aggregations | `groupBy`, `agg`, window functions (`Window`, `row_number`, `rank`, `lag`/`lead`) |
| Joins | inner / left / right / outer / semi / anti — and their shuffle cost |
| Functions | `pyspark.sql.functions` — date, string, math, `when`/`otherwise`, `coalesce` |
| Null Handling | `na.fill`, `na.drop`, `isNull` |
| UDFs vs. Built-ins | Why built-ins beat Python UDFs; intro to Pandas UDFs |
| Spark SQL | `spark.sql()`, temp views, mixing SQL with the DataFrame API |

### Datasets
* **Primary:** `samples.tpch.lineitem` — 30 million rows of order line items (TPC-H benchmark)
* **Joins:** `samples.tpch.orders` (7.5M), `customer` (750K), `supplier` (50K), `nation` (25)
* **Project:** `samples.nyctaxi.trips` — 22K real NYC taxi trips (for the hands-on exercise)

---

## FAQ: Python & PySpark Basics

### Spark DataFrame vs pandas DataFrame

| | Spark DataFrame | pandas DataFrame |
| --- | --- | --- |
| **Where it lives** | Distributed across many machines | Single machine RAM |
| **Size limit** | Petabytes (just add nodes) | \~10 GB (limited by RAM) |
| **Execution** | Lazy (builds a plan, runs on action) | Eager (runs immediately) |
| **Processing** | Parallel across 1000s of cores | Single-threaded (mostly) |
| **API** | `.select()`, `.filter()`, `.groupBy()` | `.loc[]`, `.iloc[]`, bracket indexing |
| **Mutability** | Immutable (every operation returns a NEW df) | Mutable (modify in-place) |
| **Use case** | Big data ETL, production pipelines | Small data, exploration, ML prototyping |

Both communities use the variable name `df` — they’re completely different objects.

### Why Python (not Scala/Java)?

* **Industry standard** — 80%+ of Spark jobs in production are PySpark
* **Same performance** — PySpark sends instructions to the JVM, which runs the same optimized Catalyst/Tungsten/Photon code regardless of language
* **Ecosystem** — pandas, scikit-learn, MLflow, notebooks all speak Python
* **Hiring** — most Data Engineer job postings say “PySpark” not “Scala Spark”

### Small Python things you’ll see in this notebook

| Code | What it does |
| --- | --- |
| `print("=" * 60)` | Prints 60 equals signs — a visual separator (purely cosmetic) |
| `f"Rows: {x:,}"` | F-string with comma formatting (29999795 → 29,999,795) |
| `_ = df.schema` | Underscore = throwaway variable (“I don’t care about this result”) |
| `from X import Y` | Loads specific tools from a library so you can use them |
| `time.time()` | Returns current time in seconds (for measuring duration) |

In [0]:
# ============================================================
# STEP 1: Create a schema and volume for our project
# ============================================================
# We'll use the 'arnalladatabricks' catalog (we have CREATE permission).
# Volume = a managed location to store raw files in Unity Catalog.

spark.sql("CREATE SCHEMA IF NOT EXISTS arnalladatabricks.pyspark_learning")
spark.sql("CREATE VOLUME IF NOT EXISTS arnalladatabricks.pyspark_learning.raw_data")

print("✅ Schema: arnalladatabricks.pyspark_learning")
print("✅ Volume: arnalladatabricks.pyspark_learning.raw_data")
print()
print("Volume path: /Volumes/arnalladatabricks/pyspark_learning/raw_data/")

In [0]:
# ============================================================
# STEP 2: Load our datasets
# ============================================================
# We'll use TWO datasets from the 'samples' catalog:
#
# 1. samples.tpch (TPC-H benchmark) — 30 MILLION rows in lineitem
#    Perfect for: schemas, transforms, aggregations, window functions, performance
#    Multiple related tables for join demos (orders, customer, supplier, etc.)
#
# 2. samples.nyctaxi.trips — 22K rows of real NYC taxi trips
#    Perfect for: the hands-on pipeline project at the end
#
# Why these? The external download was blocked by security policy.
# These are BETTER anyway: pre-loaded, instantly available, and genuinely large.

# --- Main dataset: TPC-H Lineitem (30M rows, 16 columns) ---
df = spark.read.table("samples.tpch.lineitem")
print(f"Main dataset: samples.tpch.lineitem")
print(f"  Rows:    {df.count():,}")
print(f"  Columns: {len(df.columns)}")

# --- Supporting tables for joins ---
df_orders = spark.read.table("samples.tpch.orders")
df_customers = spark.read.table("samples.tpch.customer")
df_suppliers = spark.read.table("samples.tpch.supplier")
df_nations = spark.read.table("samples.tpch.nation")

print(f"\nSupporting tables:")
print(f"  orders:    {df_orders.count():,} rows")
print(f"  customer:  {df_customers.count():,} rows")
print(f"  supplier:  {df_suppliers.count():,} rows")
print(f"  nation:    {df_nations.count():,} rows")

# --- NYC Taxi for the final project ---
df_taxi = spark.read.table("samples.nyctaxi.trips")
print(f"\nFinal project dataset: samples.nyctaxi.trips")
print(f"  Rows:    {df_taxi.count():,} rows")

In [0]:
# ============================================================
# STEP 3: Explore the schema and data
# ============================================================
# The TPC-H lineitem table simulates order line items for a business.
# Each row = one item in an order (quantity, price, discounts, dates, status).

print("=" * 60)
print("SCHEMA of samples.tpch.lineitem (our main dataset)")
print("=" * 60)
df.printSchema()

print("\nFirst 5 rows (scroll right to see all columns):")
df.show(5, truncate=20)

## 2.1 — Schemas: The Blueprint of Your Data

A **schema** defines the column names, data types, and nullability of a DataFrame. There are two ways to handle schemas when reading data:

| Approach | How | Pros | Cons |
| --- | --- | --- | --- |
| **inferSchema** | Spark scans the data to guess types | Zero effort, good for exploration | Slow on large files (extra pass), can guess wrong (e.g., "10" as string vs int), non-deterministic |
| **Explicit schema** | You define `StructType` upfront | Fast (no extra scan), guaranteed correctness, catches schema drift | More code to write |

### Why explicit wins in production:
1. **Performance** — No extra data scan pass to infer types
2. **Safety** — Fails immediately if data doesn't match (catches upstream bugs)
3. **Consistency** — Same schema every run, even if source data has quirks
4. **Documentation** — Schema-as-code is self-documenting

**Rule:** Use `inferSchema` for exploration. Use explicit `StructType` for pipelines.

### Deep Dive: What is StructType in Real Life?

**StructType** is a Python object that says: “my data has these columns, in this order, with these types.” It’s a **contract** between you and the raw file.

**In real projects, you rarely start with a clean table.** Data arrives as:
* CSV dumps from vendors (emailed or dropped in cloud storage)
* JSON from REST APIs
* Parquet exports from other systems

These raw files have NO table definition — just bytes on disk. Without a StructType, Spark guesses types (and guesses wrong: is `"12345"` a string or integer? Is `"2024-01-15"` a date or string?).

### What does `nullable=True` mean?

`nullable=True` means “this column is *allowed* to contain null values.” It’s a permission, not a guarantee.

**Why even primary keys are nullable=True:**
* Spark DataFrames don’t enforce primary keys (unlike SQL databases)
* `nullable=False` can crash your pipeline if a corrupted row has a null
* **Production pattern:** Set `nullable=True` (be permissive on read), then validate explicitly:
  * `df.filter(col("id").isNull()).count()` — check for problems
  * `df.na.drop(subset=["id"])` — remove bad rows

This is the **“read permissive, validate explicitly”** pattern — catch bad data gracefully instead of crashing mid-pipeline.

### All Keywords Used in Steps 1–6

| Code | What it does |
| --- | --- |
| `spark` | Pre-built SparkSession object — your entry point to everything Spark |
| `spark.sql("...")` | Runs a SQL command (CREATE, SELECT, etc.) |
| `spark.read.table("...")` | Reads an existing Unity Catalog table into a DataFrame |
| `spark.read.schema(s).parquet("path")` | Reads a Parquet file with an explicit schema |
| `.option("inferSchema", "true")` | Tells Spark “scan the data and guess types” |
| `df.count()` | ACTION — counts all rows |
| `df.columns` | Returns a Python list of column names |
| `df.printSchema()` | Prints column names, types, and nullability |
| `df.show(5)` | ACTION — displays first 5 rows |
| `.write.mode("overwrite").parquet("path")` | Save DataFrame as Parquet (overwrite if exists) |
| `spark.createDataFrame(data, schema)` | Makes a DataFrame from a Python list + schema |
| `StructType([...])` | The whole schema — a list of StructFields |
| `StructField(name, type, nullable)` | One column definition |
| `LongType()` | Big integer (up to 9 quintillion) |
| `IntegerType()` | Normal integer (-2B to +2B) |
| `DecimalType(18, 2)` | Exact decimal — 18 digits, 2 after the dot (perfect for money) |
| `StringType()` | Text |
| `DateType()` | Date only (no time) |
| `TimestampType()` | Date + time |

In [0]:
# ============================================================
# STEP 4: Defining an explicit schema with StructType
# ============================================================
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, IntegerType,
    DoubleType, DecimalType, DateType, TimestampType
)

# --- Define schema explicitly (matches the lineitem table) ---
lineitem_schema = StructType([
    StructField("l_orderkey", LongType(), nullable=True),
    StructField("l_partkey", LongType(), nullable=True),
    StructField("l_suppkey", LongType(), nullable=True),
    StructField("l_linenumber", IntegerType(), nullable=True),
    StructField("l_quantity", DecimalType(18, 2), nullable=True),
    StructField("l_extendedprice", DecimalType(18, 2), nullable=True),
    StructField("l_discount", DecimalType(18, 2), nullable=True),
    StructField("l_tax", DecimalType(18, 2), nullable=True),
    StructField("l_returnflag", StringType(), nullable=True),
    StructField("l_linestatus", StringType(), nullable=True),
    StructField("l_shipdate", DateType(), nullable=True),
    StructField("l_commitdate", DateType(), nullable=True),
    StructField("l_receiptdate", DateType(), nullable=True),
    StructField("l_shipinstruct", StringType(), nullable=True),
    StructField("l_shipmode", StringType(), nullable=True),
    StructField("l_comment", StringType(), nullable=True),
])

print("Explicit schema defined!")
print(f"Fields: {len(lineitem_schema.fields)}")
print(f"\nSchema structure:")
for field in lineitem_schema.fields:
    print(f"  {field.name:20s} {str(field.dataType):20s} nullable={field.nullable}")

In [0]:
# ============================================================
# STEP 5: Three ways to create DataFrames
# ============================================================

# --- WAY 1: From a table (most common in Databricks) ---
df_from_table = spark.read.table("samples.tpch.lineitem")
print("WAY 1: spark.read.table('catalog.schema.table')")
print(f"  Rows: {df_from_table.count():,}\n")

# --- WAY 2: From files with explicit schema (production pattern) ---
# Writing a sample to our volume first, then reading it back with schema
df.limit(10000).write.mode("overwrite").parquet(
    "/Volumes/arnalladatabricks/pyspark_learning/raw_data/lineitem_sample.parquet"
)

df_from_file = spark.read.schema(lineitem_schema).parquet(
    "/Volumes/arnalladatabricks/pyspark_learning/raw_data/lineitem_sample.parquet"
)
print("WAY 2: spark.read.schema(my_schema).parquet('path')")
print(f"  Rows: {df_from_file.count():,}")
print(f"  Schema enforced: {df_from_file.schema == lineitem_schema}\n")

# --- WAY 3: From Python data (for testing/prototyping) ---
test_data = [(1, "Alice", 100.0), (2, "Bob", 200.0)]
test_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("amount", DoubleType(), True),
])
df_from_python = spark.createDataFrame(test_data, schema=test_schema)
print("WAY 3: spark.createDataFrame(data, schema)")
df_from_python.show()

In [0]:
# ============================================================
# STEP 6: inferSchema vs explicit — speed comparison
# ============================================================
import time

parquet_path = "/Volumes/arnalladatabricks/pyspark_learning/raw_data/lineitem_sample.parquet"

# --- With inferSchema (Spark must scan data to guess types) ---
start = time.time()
df_inferred = spark.read.option("inferSchema", "true").parquet(parquet_path)
_ = df_inferred.schema  # force schema resolution
infer_time = time.time() - start

# --- With explicit schema (no scan needed) ---
start = time.time()
df_explicit = spark.read.schema(lineitem_schema).parquet(parquet_path)
_ = df_explicit.schema  # force schema resolution
explicit_time = time.time() - start

print("Speed comparison (schema resolution):")
print(f"  inferSchema: {infer_time:.3f}s")
print(f"  explicit:    {explicit_time:.3f}s")
print(f"  Winner:      {'explicit' if explicit_time < infer_time else 'infer'} "
      f"({abs(infer_time - explicit_time) / max(infer_time, 0.001) * 100:.0f}% faster)")
print()
print("On a 500 GB CSV, inferSchema does a FULL PASS just to guess types.")
print("Explicit schema skips that entirely. In production, always define your schema.")

## 2.2 — Select & Filter

### `select` — Choose columns
```python
df.select("col1", "col2")           # by name
df.select(col("col1"), col("col2"))  # by Column object
df.select(df.col1, df.col2)          # by DataFrame attribute
df.select("*")                       # all columns
```

### `filter` / `where` — Choose rows (they are identical)
```python
df.filter(df.fare > 10)              # by Column expression
df.filter("fare > 10")              # by SQL string
df.where(col("city") == "Toronto")  # .where() is an alias for .filter()
```

### `col()` — Reference a column by name
```python
from pyspark.sql.functions import col
col("my_column")  # returns a Column object you can use in expressions
```

**When to use `col()` vs `df.column`:**
* `col("x")` — works anywhere, even when you don't have a reference to the DataFrame
* `df.x` — concise, but only works when you have the DataFrame variable

### Deep Dive: Select, Filter, and .alias()

**`select` = Choose COLUMNS (vertical slicing)**

Your DataFrame has 16 columns. You rarely need all 16. `select` picks only the ones you care about — less data to shuffle, faster queries.

**`filter` = Choose ROWS (horizontal slicing)**

Out of 30 million rows, you only want ones matching a condition. `filter` keeps matches, discards the rest. Filter is CHEAP (narrow transformation, no shuffle).

**`col()` = How you “point” at a column inside expressions**

When you write `col("price") * (1 - col("discount"))`, you’re building a formula that Spark applies to every row.

**Why `&` not `and`, and why parentheses?**

Python’s `and` keyword doesn’t work with Spark Column objects. `&` is the operator Spark overloads for column logic. Parentheses are needed because `&` has higher precedence than `==`:
* `col("a") == "AIR" & col("b") > 5` → **broken** (Python groups wrong)
* `(col("a") == "AIR") & (col("b") > 5)` → **correct**

**What does `.alias()` do?**

`.alias("name")` gives a name to a computed column. Without it, Spark auto-generates ugly names like `(l_extendedprice * (1 - l_discount))`. With `.alias("revenue")`, you get a clean, readable column name.

Think of it like SQL’s `AS`: `SELECT price * qty AS total` = `(col("price") * col("qty")).alias("total")`

**The performance habit:** “Filter early, select only what you need” — always shrink your data BEFORE expensive operations (joins, groupBy).

In [0]:
# ============================================================
# STEP 7: Select & Filter in action
# ============================================================
from pyspark.sql.functions import col

# --- SELECT: Pick specific columns ---
print("SELECT specific columns:")
df.select("l_orderkey", "l_quantity", "l_extendedprice", "l_shipdate").show(5)

# --- SELECT with expressions (computed columns) ---
print("SELECT with expressions (revenue = price * (1 - discount)):")
df.select(
    col("l_orderkey"),
    col("l_extendedprice"),
    col("l_discount"),
    (col("l_extendedprice") * (1 - col("l_discount"))).alias("revenue")
).show(5)

# --- FILTER: Keep only rows matching a condition ---
print("FILTER: High-value items (price > 50000):")
df.filter(col("l_extendedprice") > 50000) \
  .select("l_orderkey", "l_extendedprice", "l_quantity", "l_shipmode") \
  .show(5)

# --- FILTER with multiple conditions ---
print("FILTER: Expensive AIR shipments with high discount:")
df.filter(
    (col("l_shipmode") == "AIR") &
    (col("l_discount") > 0.05) &
    (col("l_extendedprice") > 40000)
).select("l_orderkey", "l_shipmode", "l_discount", "l_extendedprice") \
 .show(5)

print("→ Note: use & (not 'and'), | (not 'or'), ~ (not 'not')")
print("  Each condition must be wrapped in parentheses!")

## 2.3 — Transformations

Transformations create NEW DataFrames (remember: DataFrames are immutable).

| Method | What it does |
| --- | --- |
| `withColumn(name, expr)` | Add or replace a column |
| `withColumnRenamed(old, new)` | Rename a column |
| `drop(col1, col2, ...)` | Remove columns |
| `cast(type)` | Change a column's data type |

**Key insight:** `withColumn` doesn't modify the original DataFrame — it returns a NEW one.
```python
df2 = df.withColumn("new_col", expr)  # df is unchanged; df2 has the new column
```

### Deep Dive: Why Immutability Matters

**DataFrames are IMMUTABLE.** When you do `df.withColumn("new_col", ...)`, the original `df` is untouched. You get back a brand-new DataFrame.

This is unlike pandas where `df["new_col"] = ...` modifies in-place.

```python
df_transformed = df.withColumn("revenue", ...)
# df still has original columns
# df_transformed has the new "revenue" column
```

**Real-world context for each transformation:**

| Method | When you’d use it |
| --- | --- |
| `withColumn` | Vendor sends raw `price` and `discount`; you need `revenue = price * (1-discount)` |
| `withColumnRenamed` | Vendor calls it `l_shipmode`; your team expects `ship_mode` |
| `drop` | Raw table has 50 columns; your report needs 10 — drop the rest |
| `cast` | CSV read `amount` as string (`"459.99"`); you need it as a number for math |

**Every ETL pipeline follows this flow:**
1. **Read** raw data
2. **Select** relevant columns
3. **Filter** bad rows
4. **Transform** — add derived columns, rename, cast, drop junk
5. **Write** clean output

Step 4 (transform) is where 80% of your pipeline logic lives.

In [0]:
# ============================================================
# STEP 8: withColumn, withColumnRenamed, drop, cast
# ============================================================
from pyspark.sql.functions import col, round as spark_round, datediff, lit, upper

# --- withColumn: Add a computed column ---
df_transformed = df.select(
    "l_orderkey", "l_extendedprice", "l_discount", "l_tax",
    "l_shipdate", "l_receiptdate", "l_shipmode"
).withColumn(
    "revenue", spark_round(col("l_extendedprice") * (1 - col("l_discount")), 2)
).withColumn(
    "revenue_with_tax",
    spark_round(col("l_extendedprice") * (1 - col("l_discount")) * (1 + col("l_tax")), 2)
).withColumn(
    "shipping_days", datediff(col("l_receiptdate"), col("l_shipdate"))
)

print("withColumn — added 3 derived columns:")
df_transformed.show(5)

# --- withColumnRenamed: Rename columns ---
df_renamed = df_transformed.withColumnRenamed("l_shipmode", "ship_mode") \
                           .withColumnRenamed("l_orderkey", "order_key")
print(f"withColumnRenamed: {df_renamed.columns[:3]}")

# --- drop: Remove columns you don't need ---
df_slim = df_transformed.drop("l_tax", "l_discount")
print(f"\ndrop('l_tax', 'l_discount'): {len(df_transformed.columns)} cols → {len(df_slim.columns)} cols")

# --- cast: Change data types ---
df_casted = df.select(
    col("l_quantity").cast("integer").alias("qty_int"),
    col("l_extendedprice").cast("double").alias("price_double"),
    col("l_shipdate").cast("string").alias("shipdate_string"),
)
print("\ncast — type conversions:")
df_casted.printSchema()
df_casted.show(3)

## 2.4 — Aggregations & Window Functions

### Basic Aggregation: `groupBy` + `agg`
```python
df.groupBy("category").agg(
    count("*").alias("total"),
    sum("revenue").alias("total_revenue"),
    avg("price").alias("avg_price")
)
```

### Window Functions: Compute across related rows WITHOUT collapsing them

Unlike `groupBy` (which reduces rows), window functions ADD a column while keeping all rows.

| Function | What it does |
| --- | --- |
| `row_number()` | Sequential number within each partition (1, 2, 3...) |
| `rank()` | Rank with gaps (1, 2, 2, 4) for ties |
| `dense_rank()` | Rank without gaps (1, 2, 2, 3) |
| `lag(col, n)` | Value from N rows BEFORE current row |
| `lead(col, n)` | Value from N rows AFTER current row |
| `sum().over(window)` | Running/cumulative sum within partition |

```python
from pyspark.sql.window import Window
window_spec = Window.partitionBy("category").orderBy(col("date").desc())
df.withColumn("rank", rank().over(window_spec))
```

In [0]:
# ============================================================
# STEP 9: groupBy + agg (basic aggregations)
# ============================================================
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, min as spark_min,
    max as spark_max, round as spark_round
)

# --- Revenue by ship mode ---
print("Revenue by ship mode (30M rows aggregated):")
df.groupBy("l_shipmode").agg(
    count("*").alias("order_lines"),
    spark_round(spark_sum(col("l_extendedprice") * (1 - col("l_discount"))), 2).alias("total_revenue"),
    spark_round(avg("l_quantity"), 1).alias("avg_quantity"),
    spark_round(avg("l_discount"), 3).alias("avg_discount")
).orderBy(col("total_revenue").desc()) \
 .show()

# --- Multiple groupBy columns ---
print("Revenue by return flag + ship status (like a pivot table):")
df.groupBy("l_returnflag", "l_linestatus").agg(
    count("*").alias("count"),
    spark_round(spark_sum("l_extendedprice"), 2).alias("sum_price"),
    spark_round(avg("l_extendedprice"), 2).alias("avg_price"),
).orderBy("l_returnflag", "l_linestatus") \
 .show()

In [0]:
# ============================================================
# STEP 10: Window functions (rank, row_number, lag, lead)
# ============================================================
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number, rank, dense_rank, lag, lead,
    sum as spark_sum, col, round as spark_round
)

# Work with a focused subset for clarity
df_orders_agg = df.groupBy("l_orderkey", "l_shipmode").agg(
    spark_round(spark_sum(col("l_extendedprice") * (1 - col("l_discount"))), 2).alias("order_revenue")
)

# --- Define a window: partition by ship mode, ordered by revenue descending ---
window_by_mode = Window.partitionBy("l_shipmode").orderBy(col("order_revenue").desc())

# --- row_number: sequential rank (no ties) ---
# --- rank: allows ties with gaps ---
df_ranked = df_orders_agg.withColumn(
    "row_num", row_number().over(window_by_mode)
).withColumn(
    "rnk", rank().over(window_by_mode)
).withColumn(
    "dense_rnk", dense_rank().over(window_by_mode)
)

print("Top 5 orders by revenue within each ship mode (window functions):")
df_ranked.filter(col("row_num") <= 3) \
    .orderBy("l_shipmode", "row_num") \
    .show(21, truncate=False)

# --- lag / lead: compare to previous/next row ---
window_ordered = Window.partitionBy("l_shipmode").orderBy("order_revenue")

df_with_lag = df_orders_agg.withColumn(
    "prev_revenue", lag("order_revenue", 1).over(window_ordered)
).withColumn(
    "next_revenue", lead("order_revenue", 1).over(window_ordered)
).withColumn(
    "diff_from_prev", col("order_revenue") - col("prev_revenue")
)

print("lag/lead — compare each order's revenue to previous and next:")
df_with_lag.filter(col("l_shipmode") == "AIR") \
    .orderBy("order_revenue") \
    .select("l_shipmode", "order_revenue", "prev_revenue", "next_revenue", "diff_from_prev") \
    .show(10)

### Deep Dive: How Window Functions Work

**groupBy collapses rows** (30M → 7 rows). **Window functions ADD a column while keeping every row.**

Real-world analogy: You have all sales reps’ revenue. You want to **rank** them within each region — but still see every rep’s row with a rank number added.

**Three pieces of a window:**
1. **`partitionBy`** — “Within each ___” (creates independent groups, counter resets per group)
2. **`orderBy`** — “Sorted by ___” (determines row ordering within the partition)
3. **The function** — What to compute (rank, row_number, lag, etc.)

**What is `.over(window_spec)`?**

`.over()` connects a window function to its definition. Without it, `rank()` doesn’t know what to rank or how. `.over()` gives it context: “rank within each ship mode, by revenue descending.”

**Why does the output show 1,2,3,1,2,3,1,2,3?**

`partitionBy("l_shipmode")` resets the counter for EACH ship mode. 7 ship modes × top 3 = 21 rows, each group numbered 1,2,3 independently.

**Why do rank, row_number, and dense_rank show the same values?**

Because there are **no ties** in our data (every order has a unique revenue). The difference only appears when values are equal:

| Scenario (values: 500, 400, 400, 300) | row_number | rank | dense_rank |
| --- | --- | --- | --- |
| Result | 1, 2, 3, 4 | 1, 2, 2, 4 | 1, 2, 2, 3 |

* `row_number` — always unique (breaks ties arbitrarily)
* `rank` — same for ties, then **skips** (2,2,4)
* `dense_rank` — same for ties, **no skip** (2,2,3)

**lag / lead in simple terms:**
* `lag("col", 1)` = “what was the previous row’s value?” (look backward)
* `lead("col", 1)` = “what is the next row’s value?” (look forward)

Useful for: “how much did revenue change from yesterday?” or “what’s the gap between consecutive orders?”

## 2.5 — Joins

Joins combine rows from two DataFrames based on a key. **All joins are wide transformations** (they shuffle data).

| Join Type | Returns | Use Case |
| --- | --- | --- |
| `inner` | Only rows with matching keys in BOTH sides | Default; most common |
| `left` | All rows from left + matches from right (NULLs for non-matches) | Keep all left-side records |
| `right` | All rows from right + matches from left | Keep all right-side records |
| `outer` / `full` | All rows from BOTH sides (NULLs where no match) | Union of both sides |
| `left_semi` | Rows from left that HAVE a match in right (no right columns) | EXISTS / IN subquery equivalent |
| `left_anti` | Rows from left that have NO match in right | NOT EXISTS / NOT IN equivalent |

### Shuffle Cost
* **Both sides shuffle** by default (sort-merge join) — expensive!
* **Broadcast join**: If one side is small, Spark sends it to all executors (no shuffle on that side)
* Catalyst auto-broadcasts tables < 10 MB; you can hint with `broadcast(df)`

```python
from pyspark.sql.functions import broadcast
df_big.join(broadcast(df_small), on="key")  # force broadcast of small table
```

### Deep Dive: Joins in Plain English

**Real-world scenario:** You have an Orders table and a Customers table. You want one table showing order details + customer name. Joins match rows by a key (`customer_id`).

**The 6 types in plain English:**

| Type | Plain English |
| --- | --- |
| **inner** | “Only rows matching in BOTH tables” |
| **left** | “All rows from left; attach right where possible (NULL if no match)” |
| **right** | “Same but keep all from right” (rarely used — just swap tables and use left) |
| **outer** | “Everything from both; NULLs where no match” |
| **left_semi** | “Left rows that HAVE a match — no right columns added” (like SQL `WHERE EXISTS`) |
| **left_anti** | “Left rows that have NO match” (like SQL `WHERE NOT EXISTS`) |

**Why joins are expensive:** To match keys, both rows must be on the **same executor**. If they’re on different machines, Spark shuffles data across the network. Two tables = potentially two shuffles.

**Broadcast join — the optimization:** If one table is small (e.g., 25 nations), Spark COPIES it to every executor. The big table doesn’t move at all. `broadcast(df_small)` forces this.

Spark auto-broadcasts tables < 10 MB. Use `broadcast()` to force it for larger-but-still-manageable tables.

In [0]:
# ============================================================
# STEP 11: All 6 join types demonstrated
# ============================================================
from pyspark.sql.functions import broadcast, col

# --- Join lineitem with orders (30M → 7.5M) ---
print("="* 60)
print("INNER JOIN: lineitem + orders (on l_orderkey = o_orderkey)")
print("="* 60)
df_joined = df.join(df_orders, df.l_orderkey == df_orders.o_orderkey, "inner")
print(f"Result: {df_joined.count():,} rows")
df_joined.select(
    "l_orderkey", "l_extendedprice", "l_shipmode",
    "o_orderstatus", "o_orderdate", "o_orderpriority"
).show(5)

# --- LEFT JOIN: orders + customers (show order with customer info) ---
print("LEFT JOIN: orders + customers (on o_custkey = c_custkey)")
df_orders_cust = df_orders.join(
    df_customers,
    df_orders.o_custkey == df_customers.c_custkey,
    "left"
)
df_orders_cust.select(
    "o_orderkey", "o_orderdate", "o_totalprice", "c_name", "c_mktsegment"
).show(5)

# --- LEFT SEMI: Orders that HAVE line items with AIR shipping ---
print("LEFT SEMI: Orders that have at least one AIR shipment (no right cols):")
df_air = df.filter(col("l_shipmode") == "AIR").select("l_orderkey")
df_orders.join(df_air, df_orders.o_orderkey == df_air.l_orderkey, "left_semi") \
    .select("o_orderkey", "o_orderdate", "o_totalprice") \
    .show(5)

# --- LEFT ANTI: Orders with NO line items over $50K ---
print("LEFT ANTI: Orders with no high-value line items (anti-join):")
df_expensive = df.filter(col("l_extendedprice") > 50000).select("l_orderkey")
df_orders.join(df_expensive, df_orders.o_orderkey == df_expensive.l_orderkey, "left_anti") \
    .select("o_orderkey", "o_orderdate", "o_totalprice") \
    .show(5)

# --- BROADCAST JOIN: Small table (nations) broadcast to all executors ---
print("BROADCAST JOIN: suppliers + nations (broadcast the 25-row table):")
df_supp_nation = df_suppliers.join(
    broadcast(df_nations),
    df_suppliers.s_nationkey == df_nations.n_nationkey,
    "inner"
)
df_supp_nation.select("s_name", "s_phone", "n_name").show(5)
print("→ broadcast() avoids shuffling the small table — much faster!")

## 2.6 — Functions & Null Handling

### The `pyspark.sql.functions` module

This is your toolbox — 200+ functions that Catalyst can optimize (unlike Python UDFs).

| Category | Key Functions |
| --- | --- |
| **Math** | `abs`, `round`, `ceil`, `floor`, `sqrt`, `log`, `pow` |
| **String** | `upper`, `lower`, `trim`, `length`, `substring`, `concat`, `regexp_replace`, `split` |
| **Date/Time** | `year`, `month`, `dayofmonth`, `datediff`, `date_add`, `date_format`, `current_date` |
| **Conditional** | `when`/`otherwise` (like SQL CASE WHEN), `coalesce` |
| **Aggregate** | `count`, `sum`, `avg`, `min`, `max`, `countDistinct`, `collect_list` |
| **Null** | `isNull`, `isNotNull`, `coalesce`, `na.fill`, `na.drop` |

### Null Handling
```python
df.na.drop()                      # drop rows with ANY null
df.na.drop(subset=["col1"])       # drop rows where col1 is null
df.na.fill(0)                     # fill all nulls with 0
df.na.fill({"col1": 0, "col2": "unknown"})  # different fill per column
df.filter(col("x").isNotNull())   # filter out nulls
```

### Deep Dive: Functions & Null Handling

**`when` / `.otherwise`** = If/elif/else for columns (SQL’s CASE WHEN):

Checks conditions top to bottom, returns first match. `.otherwise()` is the default if nothing matches.

Real-world use: “Categorize every shipment as Express/Standard/Economy based on delivery days.”

**Date functions — why they matter:**
* `year()` + `month()` → “group sales by month”
* `datediff()` → “calculate SLA violations”
* `date_format(col, "EEEE")` → “which day of week has most orders?”

**String functions — for messy vendor data:**
* `upper()`/`lower()` → standardize case
* `trim()` → remove extra spaces
* `length()` → find suspiciously short/long values

**Null handling — the 4 tools:**

| Tool | What it does | When to use |
| --- | --- | --- |
| `na.drop(subset=[...])` | DELETE rows with nulls | Remove incomplete records |
| `na.fill({"col": value})` | REPLACE nulls with defaults | Missing discount = 0 |
| `coalesce(col1, col2)` | Pick first NON-NULL from a list | “Use discount, or fallback to 0” |
| `col("x").isNull()` | CHECK if null (for filtering) | “Show me rows with missing data” |

**What is `lit()`?** Creates a “literal” constant value as a Column. When you write `coalesce(col("discount"), lit(0.0))`, `lit(0.0)` turns the Python number `0.0` into a Spark Column that can be used in expressions.

**Production pattern:** Read raw → count nulls per column → decide (drop/fill/keep) → document choices.

In [0]:
# ============================================================
# STEP 12: pyspark.sql.functions showcase
# ============================================================
from pyspark.sql.functions import (
    col, when, coalesce, lit,
    year, month, dayofmonth, datediff, date_format,
    upper, lower, length, substring, concat, concat_ws,
    round as spark_round, abs as spark_abs,
    count, sum as spark_sum, avg, countDistinct
)

# --- CONDITIONAL: when / otherwise (like SQL CASE WHEN) ---
print("WHEN/OTHERWISE — categorize shipping speed:")
df_speed = df.select(
    "l_orderkey", "l_shipdate", "l_receiptdate", "l_shipmode"
).withColumn(
    "ship_days", datediff(col("l_receiptdate"), col("l_shipdate"))
).withColumn(
    "speed_category",
    when(col("ship_days") <= 5, "Express")
    .when(col("ship_days") <= 14, "Standard")
    .when(col("ship_days") <= 30, "Economy")
    .otherwise("Delayed")
)
df_speed.show(10)

# Count by category
print("Distribution of speed categories across 30M shipments:")
df_speed.groupBy("speed_category").count() \
    .orderBy(col("count").desc()).show()

# --- DATE FUNCTIONS ---
print("DATE FUNCTIONS — extract year, month, format:")
df.select(
    col("l_shipdate"),
    year("l_shipdate").alias("ship_year"),
    month("l_shipdate").alias("ship_month"),
    dayofmonth("l_shipdate").alias("ship_day"),
    date_format("l_shipdate", "EEEE").alias("day_of_week"),
).show(5)

# --- STRING FUNCTIONS ---
print("STRING FUNCTIONS:")
df.select(
    col("l_shipinstruct"),
    upper("l_shipinstruct").alias("upper"),
    lower("l_shipinstruct").alias("lower"),
    length("l_shipinstruct").alias("len"),
    substring("l_shipinstruct", 1, 4).alias("first_4"),
).distinct().show()

In [0]:
# ============================================================
# STEP 13: Null handling
# ============================================================
from pyspark.sql.functions import col, count, when, isnan, coalesce, lit

# The TPC-H data is clean (no nulls), so let's create some to demonstrate
df_with_nulls = df.select(
    "l_orderkey", "l_quantity", "l_discount", "l_shipmode", "l_comment"
).withColumn(
    "nullable_discount",
    when(col("l_discount") == 0, None).otherwise(col("l_discount"))
).withColumn(
    "nullable_comment",
    when(length(col("l_comment")) < 15, None).otherwise(col("l_comment"))
)

# --- Check for nulls ---
print("NULL counts per column:")
df_with_nulls.select(
    count("*").alias("total_rows"),
    count("nullable_discount").alias("non_null_discount"),
    (count("*") - count("nullable_discount")).alias("null_discount"),
    (count("*") - count("nullable_comment")).alias("null_comment"),
).show()

# --- na.fill: Replace nulls ---
print("na.fill — replace nulls with defaults:")
df_filled = df_with_nulls.na.fill({
    "nullable_discount": 0.0,
    "nullable_comment": "NO COMMENT"
})
df_filled.filter(col("nullable_comment") == "NO COMMENT").show(3)

# --- na.drop: Remove rows with nulls ---
original_count = df_with_nulls.count()
dropped_count = df_with_nulls.na.drop(subset=["nullable_discount"]).count()
print(f"na.drop(subset=['nullable_discount']):")
print(f"  Before: {original_count:,} rows")
print(f"  After:  {dropped_count:,} rows")
print(f"  Dropped: {original_count - dropped_count:,} rows")

# --- coalesce: Pick first non-null value ---
print("\ncoalesce — pick first non-null:")
df_with_nulls.select(
    col("l_orderkey"),
    col("nullable_discount"),
    coalesce(col("nullable_discount"), lit(0.0)).alias("discount_or_zero")
).filter(col("nullable_discount").isNull()).show(5)

## 2.7 — UDFs vs. Built-in Functions

### Why built-ins always win:

| | Built-in Functions | Python UDFs | Pandas UDFs |
| --- | --- | --- | --- |
| **Speed** | Fastest (runs in JVM, Photon-optimized) | 10-100x slower | 3-10x slower than built-in |
| **Catalyst** | Fully optimized | Opaque black box | Partially optimized |
| **Serialization** | None (stays in JVM) | Row-by-row Python↔JVM | Batch (Arrow format) |
| **Use when** | Always first choice | Truly custom logic with no built-in equivalent | Vectorized custom logic, pandas libraries |

### The UDF penalty
```
Built-in:  data stays in JVM → Photon → fast
Python UDF: JVM → serialize → Python → compute → serialize → JVM (per row!)
Pandas UDF: JVM → Arrow batch → Python/pandas → Arrow batch → JVM (per batch)
```

**Rule:** Reach for `pyspark.sql.functions` first. Only use UDFs when there's genuinely no built-in alternative.

### Deep Dive: The Serialization Wall

**What is a UDF?** A custom Python function you register with Spark to use on DataFrame columns.

**The core problem:** Spark processes data in the **JVM** (Java). Your Python code runs in a **separate process**. With a Python UDF, every row must:
1. Leave JVM → serialize to bytes
2. Cross the bridge to Python
3. Your function runs on that ONE row
4. Result crosses back to JVM → deserialize

This happens **30 million times** (once per row).

**Factory analogy:**
* **Built-in** = a machine ON the conveyor belt stamps each item as it passes. Never stops.
* **Pandas UDF** = every 10,000 items, a box is sent to another building, processed, sent back.
* **Python UDF** = EACH item is individually carried to another building, processed, returned. 30 million trips.

**When you’d actually NEED a UDF:**
* Calling an external API per row (geocoding, ML inference)
* Using a Python library with no Spark equivalent
* Business logic so specific no built-in combination works

**But first, always check:** Can I do this with `when`/`otherwise`, `regexp_replace`, `expr()`, or a combination of built-ins? 95% of the time, yes.

**Note on the speed test:** On shared clusters (Spark Connect / USER_ISOLATION), the network overhead of Spark Connect masks the UDF penalty. On a dedicated cluster with 100M+ rows, the gap becomes 10-100x. The architecture penalty is real even if this benchmark can’t demonstrate it here.

In [0]:
# ============================================================
# STEP 14: UDF vs Built-in — COMPLEX task to reveal the real gap
# ============================================================
# Simple tasks (like upper()) don't show the gap because Spark Connect
# overhead dominates. Let's use a COMPLEX task: compute revenue with
# multiple conditions, string parsing, and math — per row.
import time
from pyspark.sql.functions import (
    udf, col, when, length, substring, round as spark_round,
    pandas_udf
)
from pyspark.sql.types import DoubleType
import pandas as pd
import numpy as np

# --- THE TASK: Complex per-row business logic ---
# For each line item, compute a "priority score" based on:
# 1. Revenue (price * (1-discount))
# 2. Bonus if shipped by AIR (+10%)
# 3. Penalty if comment contains 'special' (-5%)
# 4. Apply a non-linear transform (log)

import math

# --- PYTHON UDF: row-by-row serialization ---
@udf(returnType=DoubleType())
def priority_score_udf(price, discount, shipmode, comment):
    if price is None or discount is None:
        return 0.0
    revenue = float(price) * (1.0 - float(discount))
    if shipmode == "AIR":
        revenue *= 1.10
    if comment and "special" in comment.lower():
        revenue *= 0.95
    return math.log1p(revenue)

# --- PANDAS UDF: batched via Arrow ---
@pandas_udf(DoubleType())
def priority_score_pandas_udf(
    price: pd.Series, discount: pd.Series,
    shipmode: pd.Series, comment: pd.Series
) -> pd.Series:
    revenue = price.astype(float) * (1.0 - discount.astype(float))
    revenue = revenue.where(shipmode != "AIR", revenue * 1.10)
    has_special = comment.str.lower().str.contains("special", na=False)
    revenue = revenue.where(~has_special, revenue * 0.95)
    return np.log1p(revenue)

# --- BUILT-IN: same logic with pyspark.sql.functions ---
def priority_score_builtin(df_in):
    from pyspark.sql.functions import log1p, lower as spark_lower
    return df_in.withColumn(
        "revenue_base", col("l_extendedprice") * (1 - col("l_discount"))
    ).withColumn(
        "revenue_adj",
        when(col("l_shipmode") == "AIR", col("revenue_base") * 1.10)
        .otherwise(col("revenue_base"))
    ).withColumn(
        "revenue_adj",
        when(spark_lower(col("l_comment")).contains("special"), col("revenue_adj") * 0.95)
        .otherwise(col("revenue_adj"))
    ).withColumn(
        "score", log1p(col("revenue_adj"))
    )

# --- USE 5M ROWS to make the difference visible ---
df_test = df.select(
    "l_extendedprice", "l_discount", "l_shipmode", "l_comment"
).limit(5_000_000)

# Warmup
df_test.count()

# --- TIME: Built-in ---
start = time.time()
priority_score_builtin(df_test).select("score").count()
builtin_time = time.time() - start

# --- TIME: Python UDF ---
start = time.time()
df_test.withColumn(
    "score", priority_score_udf("l_extendedprice", "l_discount", "l_shipmode", "l_comment")
).select("score").count()
udf_time = time.time() - start

# --- TIME: Pandas UDF ---
start = time.time()
df_test.withColumn(
    "score", priority_score_pandas_udf("l_extendedprice", "l_discount", "l_shipmode", "l_comment")
).select("score").count()
pandas_udf_time = time.time() - start

print("="*60)
print("COMPLEX TASK: Priority score on 5M rows")
print("(revenue calc + conditional bonuses + log transform)")
print("="*60)
print(f"  Built-in functions: {builtin_time:.2f}s")
print(f"  Python UDF:         {udf_time:.2f}s  ({udf_time/builtin_time:.1f}x slower)")
print(f"  Pandas UDF:         {pandas_udf_time:.2f}s  ({pandas_udf_time/builtin_time:.1f}x slower)")
print()
if udf_time > builtin_time * 1.5:
    print("→ NOW you see the real gap! Complex logic + more rows = UDF penalty is clear.")
else:
    print("→ On Spark Connect (shared clusters), overhead partially masks the gap.")
    print("  On a dedicated cluster with 100M+ rows, Python UDFs are 10-100x slower.")
print()
print("KEY TAKEAWAY: The more complex your logic and the more rows you have,")
print("the bigger the UDF penalty. Built-ins stay fast because they never leave the JVM.")

## 2.8 — Spark SQL & Temp Views

You can write SQL directly in Spark! The query goes through the same Catalyst optimizer — identical performance to DataFrame API.

### Key pattern: Create a temp view, then query with SQL
```python
df.createOrReplaceTempView("my_table")
result = spark.sql("SELECT * FROM my_table WHERE price > 100")
```

### When to use SQL vs DataFrame API?

| SQL | DataFrame API |
| --- | --- |
| Complex analytical queries (CTEs, subqueries) | Multi-step ETL pipelines |
| Ad-hoc exploration | Programmatic logic (loops, conditions) |
| Team is SQL-fluent | Need Python for ML/custom logic |
| Reading existing SQL queries | Type-safe column references |

**They are equivalent** — use whichever is clearer for your use case. You can freely mix them.

In [0]:
# ============================================================
# STEP 15: Spark SQL — temp views and mixing SQL with DataFrames
# ============================================================

# --- Create temp views from our DataFrames ---
df.createOrReplaceTempView("lineitem")
df_orders.createOrReplaceTempView("orders")
df_customers.createOrReplaceTempView("customers")
df_nations.createOrReplaceTempView("nations")
df_suppliers.createOrReplaceTempView("suppliers")
print("✅ Created temp views: lineitem, orders, customers, nations, suppliers\n")

# --- Simple SQL query ---
print("SQL: Top 5 ship modes by revenue")
spark.sql("""
    SELECT 
        l_shipmode,
        COUNT(*) as num_items,
        ROUND(SUM(l_extendedprice * (1 - l_discount)), 2) as total_revenue
    FROM lineitem
    GROUP BY l_shipmode
    ORDER BY total_revenue DESC
    LIMIT 5
""").show()

# --- Complex SQL with CTE and JOIN ---
print("SQL: Top 10 customers by total spending (CTE + JOIN):")
df_top_customers = spark.sql("""
    WITH customer_spending AS (
        SELECT 
            c.c_name,
            c.c_mktsegment,
            n.n_name as nation,
            ROUND(SUM(o.o_totalprice), 2) as total_spent
        FROM customers c
        JOIN orders o ON c.c_custkey = o.o_custkey
        JOIN nations n ON c.c_nationkey = n.n_nationkey
        GROUP BY c.c_name, c.c_mktsegment, n.n_name
    )
    SELECT * FROM customer_spending
    ORDER BY total_spent DESC
    LIMIT 10
""")
df_top_customers.show(truncate=False)

# --- Mix: SQL result into DataFrame chain ---
print("MIXING SQL + DataFrame API:")
result = spark.sql("SELECT * FROM lineitem WHERE l_shipmode = 'AIR'") \
    .groupBy("l_returnflag") \
    .agg({"l_extendedprice": "sum", "l_quantity": "avg"}) \
    .withColumnRenamed("sum(l_extendedprice)", "total_price") \
    .withColumnRenamed("avg(l_quantity)", "avg_qty")
result.show()
print("→ SQL and DataFrame API produce identical execution plans.")
print("  Use whichever reads better for your use case!")

## Phase 2 Checkpoint

**Can you do these?**

- [ ] Define an explicit `StructType` schema and explain why it beats `inferSchema` in production
- [ ] Use `select`, `filter`, `withColumn`, `drop`, and `cast` fluently
- [ ] Write a `groupBy` + `agg` with multiple aggregate functions
- [ ] Use window functions (`row_number`, `rank`, `lag`/`lead`) to compare rows within groups
- [ ] Write any of the 6 join types and explain when to use `left_semi` vs `left_anti`
- [ ] Use `when`/`otherwise` for conditional logic (like SQL CASE WHEN)
- [ ] Handle nulls with `na.fill`, `na.drop`, `coalesce`, and `isNull`
- [ ] Explain why built-in functions are 10-100x faster than Python UDFs
- [ ] Mix SQL and DataFrame API using temp views and `spark.sql()`

**Golden rule:** Reach for `pyspark.sql.functions` FIRST. Only use UDFs when no built-in exists.

---

### 🛠 Your Turn: Hands-On Pipeline Project

Using `samples.nyctaxi.trips` (22K real taxi trips), build a complete pipeline:

1. **Load** the data with an explicit schema
2. **Clean** — drop nulls, remove outliers (negative fares, zero distance)
3. **Derive** — add columns: trip duration (minutes), speed (mph), fare per mile, time of day bucket
4. **Aggregate** — average fare, distance, and duration by pickup zip + time bucket
5. **Write** the result to `arnalladatabricks.pyspark_learning.taxi_summary` as a Delta table

This exercises every skill from Phase 2. If you can do this without looking at the examples above, you've mastered the DataFrame API.

---
**Next up: Phase 3 — Reading & Writing Data (file formats, Delta Lake, partitioning, optimization)**